<a href="https://colab.research.google.com/github/aw920h/thermosat/blob/main/anomaly_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install geemap
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.4 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='<your-cloud-project-id(example: civic-access-291607)>')

In [ ]:
# 1. INPUTS (update them as per requirement)
location_name = 'Barauni IOCL Refinery'  # update the site name
lon = 86.0888 #modify it
lat = 25.4385 #modify it
point_of_interest = ee.Geometry.Point([lon, lat])
roi = point_of_interest.buffer(10000) #modify it if required
# Far-field reference area: Upstream (west) on Ganges river, over water
refArea = ee.Geometry.Rectangle([lon - 0.1, lat - 0.05, lon - 0.05, lat + 0.05])  # ~5-10km west; adjust if not over river
# Options
useFarField = True
ndwiThreshold = 0.05

In [ ]:
# 2. LATEST IMAGE
col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')) \
    .merge(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')) \
    .merge(ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')) \
    .filterBounds(roi) \
    .filter(ee.Filter.lt('CLOUD_COVER', 20)) \
    .sort('system:time_start', False)

latestImage = col.first()
imageDate = ee.Date(latestImage.get('system:time_start')).format('YYYY-MM-DD')
print('Latest Image Date:', imageDate.getInfo())

Latest Image Date: 2026-01-15


In [ ]:
# 3. PROCESS IMAGE
def processImage(image):
    hasB10 = image.bandNames().contains('ST_B10')
    thermalBand = ee.String(ee.Algorithms.If(hasB10, 'ST_B10', 'ST_B6'))
    thermal = image.select(thermalBand).multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    ndwi = image.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    optical = image.select(['SR_B4', 'SR_B3', 'SR_B2']).multiply(0.0000275).add(-0.2)
    mask = ndwi.gt(ndwiThreshold)
    return image.addBands(thermal).addBands(ndwi).addBands(optical).updateMask(mask)

processed = processImage(latestImage)

In [ ]:
# 4. ANOMALY CALCULATION
if useFarField:
    refDict = processed.select('LST').reduceRegion(
        reducer=ee.Reducer.mean(), geometry=refArea, scale=100, bestEffort=True
    )
    refMean = ee.Number(refDict.get('LST'))
    refMean = ee.Algorithms.If(refMean, refMean, 0)  # Fallback if no valid pixels
    print('Ref Mean (should be ~10-20°C if over water):', refMean.getInfo())  # Debug
    deltaT = processed.select('LST').subtract(ee.Image.constant(refMean)).rename('deltaT')

    # Print stats
    stats = deltaT.reduceRegion(
        reducer=ee.Reducer.minMax(), geometry=roi, scale=100, bestEffort=True
    )
    print('DeltaT Stats:', stats.getInfo())

Ref Mean (should be ~10-20°C if over water): 17.177744585055777
DeltaT Stats: {'deltaT_max': -0.9291550650557809, 'deltaT_min': -0.9291550650557809}


In [ ]:
# 5. SETUP MAP WITH SPLIT SCREEN
Map = geemap.Map(center=[lat, lon], zoom=13)
Map.add_basemap('SATELLITE')

# Left side: Optical image
left_layer = geemap.ee_tile_layer(processed.select(['SR_B4', 'SR_B3', 'SR_B2']), {'min': 0, 'max': 0.3}, 'Optical')

# Right side: DeltaT
visParams = {
    'min': -5, 'max': 10,
    'palette': ['0000FF', '00FFFF', '00FF00', 'FFFF00', 'FF8000', 'FF0000'],
    'opacity': 0.7
}
right_layer = geemap.ee_tile_layer(deltaT, visParams, 'Thermal Hotspots')

Map.split_map(left_layer, right_layer)

# Add layers
Map.addLayer(roi, {'color': 'white'}, 'ROI Boundary')
Map.addLayer(point_of_interest, {'color': 'black'}, 'Plant Location')
if useFarField:
    Map.addLayer(refArea, {'color': 'blue'}, 'Reference Area')

# Add legend (geemap has a legend function)
legend_dict = {
    '> +5°C (Intense Discharge)': '#FF0000',
    '+3°C (Discharge)': '#FF8000',
    '+1.5°C (Mixing)': '#FFFF00',
    '0°C (Ambient)': '#00FF00',
    '-2°C (Cooling)': '#00FFFF',
    '< -5°C (Cold Anomalies)': '#0000FF'
}
Map.add_legend(title='Temp Anomaly (°C)', legend_dict=legend_dict, position='bottomright')

# Display the map
Map

Map(center=[25.4385, 86.0888], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…